In [ ]:
from langgraph.graph import StateGraph , START , END
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage , HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langgraph.checkpoint.postgres import PostgresSaver
import os
DATABASE_URL = os.getenv("DATABASE_URL")

/home/amir/LangGraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatGroq(model_name="llama-3.3-70b-versatile")



In [3]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage] , add_messages]

In [4]:
def chat_node(state : ChatState):
    messages = state['messages']

    response = model.invoke(messages)

    return {'messages' : [response]}

In [5]:
builder = StateGraph(ChatState)

builder.add_node("chat_node", chat_node)

builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)


with PostgresSaver.from_conn_string(DATABASE_URL) as checkpointer:
    checkpointer.setup()

    chatbot = builder.compile(checkpointer=checkpointer)

    thread_id = "1"

    while True:
        message = input("You: ")

        if message.lower() in ["exit", "quit", "bye"]:
            break

        config = {
            "configurable": {
                "thread_id": thread_id
            }
        }

        result = chatbot.invoke(
            {
                "messages": [HumanMessage(content=message)]
            },
            config=config,
        )

        print("AI:", result["messages"][-1].content)

AI: Your name is Amir, and you're from Nepal. I remember!
